# 09 — Informed Compression: Synthesis

**Objective**: Use mechanistic insights from notebooks 04-08 to design an **informed compression strategy** that outperforms blind uniform SVD.

**Pipeline**:
1. Load all previous analysis results (probing, patching, head ablation, neuron analysis)
2. Design per-layer, per-component SVD rank allocation based on mechanistic importance
3. Grand comparison: Uniform vs Adaptive (energy-based) vs Informed compression
4. Pareto frontier: F1 vs compression ratio for all strategies
5. Fine-tuning recovery analysis: which emotions recover and which don't?
6. Inference benchmarks (latency/throughput)
7. Final summary and recommendations

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/transformer-structural-compression-v2'
    os.chdir(PROJECT_ROOT)
    sys.path.insert(0, PROJECT_ROOT)
    !pip install -q transformers datasets scikit-learn
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    sys.path.insert(0, PROJECT_ROOT)

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import copy
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
from transformers import Trainer, TrainingArguments

from src.data import load_goemotions
from src.models import load_bert_classifier
from src.compression import (
    apply_svd_compression, get_target_layer_names, filter_layer_names,
    compute_singular_value_energy, compute_adaptive_ranks,
    count_parameters, get_compression_ratio
)
from src.utils import compute_metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# Load data and baseline model
train_ds, val_ds, test_ds, emotion_names, data_collator = load_goemotions()

MODEL_PATH = os.path.join(PROJECT_ROOT, 'results', 'bert-goemotions-final')
baseline_model = load_bert_classifier(model_path=MODEL_PATH)
baseline_model.to(device)
baseline_model.eval()

print(f"Baseline model loaded")
print(f"Test set: {len(test_ds)} samples")

## 1. Load Mechanistic Analysis Results

In [ ]:
results_dir = os.path.join(PROJECT_ROOT, 'results')

# Load probing results (from NB04)
try:
    crystallization_df = pd.read_csv(os.path.join(results_dir, 'crystallization_layers.csv'))
    probe_results = pd.read_csv(os.path.join(results_dir, 'probe_results.csv'))
    print(f"Probing results loaded: {len(crystallization_df)} emotions")
    print(f"  Mean crystallization layer: {crystallization_df['crystallization_layer'].mean():.1f}")
except FileNotFoundError:
    print("WARNING: Probing results not found. Run notebook 04 first.")
    crystallization_df = None
    probe_results = None

# Load head ablation results (from NB06)
try:
    head_importance = np.load(os.path.join(results_dir, 'head_importance_matrix.npy'))
    head_categories = pd.read_csv(os.path.join(results_dir, 'head_categories.csv'))
    layer_attn_importance = pd.read_csv(os.path.join(results_dir, 'layer_attention_importance.csv'))
    print(f"Head ablation results loaded: {head_importance.shape}")
except FileNotFoundError:
    print("WARNING: Head ablation results not found. Run notebook 06 first.")
    head_importance = None
    head_categories = None
    layer_attn_importance = None

# Load activation patching results (from NB05)
try:
    patching_results = pd.read_csv(os.path.join(results_dir, 'patching_results.csv'))
    print(f"Activation patching results loaded: {len(patching_results)} rows")
except FileNotFoundError:
    print("WARNING: Patching results not found. Run notebook 05 first.")
    patching_results = None

# Load neuron analysis results (from NB07)
try:
    neuron_selectivity = np.load(os.path.join(results_dir, 'neuron_selectivity.npy'))
    print(f"Neuron selectivity loaded: {neuron_selectivity.shape}")
except FileNotFoundError:
    print("WARNING: Neuron results not found. Run notebook 07 first.")
    neuron_selectivity = None

## 2. Design Informed Compression Strategy

The key insight: different layers and components have different importance for emotion processing. We assign SVD ranks based on:

- **Probing**: Layers where emotions crystallize need higher ranks
- **Head ablation**: Layers with critical heads need preserved attention ranks
- **Activation patching**: Layers with highest restoration effect are bottlenecks
- **Neuron selectivity**: FFN layers with highly selective neurons need preserved FFN ranks

In [ ]:
def compute_layer_importance_scores(crystallization_df=None, layer_attn_importance=None,
                                     patching_results=None, neuron_selectivity=None):
    """Compute a composite importance score per layer from all analyses.
    
    Returns:
        attn_importance: (12,) attention importance per layer (0 to 1)
        ffn_importance: (12,) FFN importance per layer (0 to 1)
    """
    n_layers = 12
    attn_scores = np.ones(n_layers) * 0.5  # default moderate
    ffn_scores = np.ones(n_layers) * 0.5
    
    n_signals = 0
    
    # Signal 1: Probing — layers near crystallization are critical
    if crystallization_df is not None:
        crystal_layers = crystallization_df['crystallization_layer'].values
        # Count how many emotions crystallize at each layer
        crystal_counts = np.zeros(13)  # layers 0-12 (including embedding)
        for cl in crystal_layers:
            if not np.isnan(cl):
                crystal_counts[int(cl)] += 1
        # Map to encoder layers (1-12 -> 0-11)
        encoder_crystal = crystal_counts[1:]  # (12,)
        # Also mark layers just BEFORE crystallization as important (they do the work)
        before_crystal = np.zeros(n_layers)
        for cl in crystal_layers:
            if not np.isnan(cl) and int(cl) >= 2:
                before_crystal[int(cl) - 2] += 0.5
                before_crystal[int(cl) - 1] += 1.0
        
        combined = encoder_crystal + before_crystal
        if combined.max() > 0:
            probing_signal = combined / combined.max()
            attn_scores += probing_signal
            ffn_scores += probing_signal
            n_signals += 1
            print(f"  Probing signal: peak at layers {np.argsort(probing_signal)[::-1][:3]}")
    
    # Signal 2: Head ablation — layers with important heads
    if layer_attn_importance is not None:
        attn_signal = layer_attn_importance['mean_head_importance'].values
        if attn_signal.max() > 0:
            attn_signal = attn_signal / attn_signal.max()
            attn_scores += attn_signal * 1.5  # weight attention signal higher for attn components
            n_signals += 1
            print(f"  Head ablation signal: peak at layers {np.argsort(attn_signal)[::-1][:3]}")
    
    # Signal 3: Activation patching (if available)
    if patching_results is not None:
        try:
            # Look for attention-only and FFN-only patching results
            for patch_type, target_scores in [('attention_only', attn_scores), ('ffn_only', ffn_scores)]:
                subset = patching_results[patching_results['patch_type'] == patch_type]
                if len(subset) > 0:
                    layer_restoration = subset.groupby('layer')['f1_restoration'].mean().values
                    if len(layer_restoration) == n_layers and layer_restoration.max() > 0:
                        patch_signal = layer_restoration / layer_restoration.max()
                        target_scores += patch_signal
                        print(f"  Patching ({patch_type}): peak at layers {np.argsort(patch_signal)[::-1][:3]}")
            n_signals += 1
        except Exception as e:
            print(f"  Patching signal skipped: {e}")
    
    # Signal 4: Neuron selectivity (for FFN)
    if neuron_selectivity is not None:
        # neuron_selectivity shape: (12, 3072, 28) or similar
        try:
            # Mean absolute selectivity per layer
            layer_selectivity = np.abs(neuron_selectivity).mean(axis=(1, 2))  # (12,)
            if layer_selectivity.max() > 0:
                neuron_signal = layer_selectivity / layer_selectivity.max()
                ffn_scores += neuron_signal * 1.5  # weight neuron signal higher for FFN
                n_signals += 1
                print(f"  Neuron selectivity: peak at layers {np.argsort(neuron_signal)[::-1][:3]}")
        except Exception as e:
            print(f"  Neuron signal skipped: {e}")
    
    # Normalize to [0, 1]
    if attn_scores.max() > 0:
        attn_scores = attn_scores / attn_scores.max()
    if ffn_scores.max() > 0:
        ffn_scores = ffn_scores / ffn_scores.max()
    
    print(f"\n  Combined from {n_signals} signal sources")
    return attn_scores, ffn_scores


print("Computing layer importance from mechanistic analyses...")
attn_importance, ffn_importance = compute_layer_importance_scores(
    crystallization_df, layer_attn_importance, patching_results, neuron_selectivity
)

In [ ]:
# Visualize importance scores
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(12)
width = 0.35
ax.bar(x - width/2, attn_importance, width, label='Attention', color='steelblue', alpha=0.8)
ax.bar(x + width/2, ffn_importance, width, label='FFN', color='coral', alpha=0.8)

ax.set_xlabel('Layer')
ax.set_ylabel('Importance Score (normalized)')
ax.set_title('Layer Importance from Mechanistic Analyses')
ax.set_xticks(x)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'informed_layer_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def design_informed_ranks(attn_importance, ffn_importance, 
                          target_compression_ratio=0.5,
                          min_rank=32, max_rank=700):
    """Design per-layer, per-component SVD ranks based on importance scores.
    
    Higher importance -> higher rank (less compression).
    Lower importance -> lower rank (more compression).
    
    Args:
        attn_importance: (12,) normalized attention importance
        ffn_importance: (12,) normalized FFN importance
        target_compression_ratio: target overall ratio (params_compressed / params_original)
        min_rank: minimum SVD rank
        max_rank: maximum SVD rank (768 is full rank for BERT)
    
    Returns:
        rank_dict: {layer_name: rank} for apply_svd_compression
    """
    # Get all target layer names
    temp_model = load_bert_classifier(model_path=MODEL_PATH)
    all_layers = get_target_layer_names(temp_model)
    del temp_model
    
    # Map importance to ranks using linear interpolation
    # importance 1.0 -> max_rank, importance 0.0 -> min_rank
    rank_dict = {}
    
    for layer_name in all_layers:
        # Extract layer index from name like "bert.encoder.layer.3.attention.self.query"
        parts = layer_name.split('.')
        layer_idx = int(parts[3])  # "bert.encoder.layer.{idx}..."
        
        # Determine if attention or FFN component
        is_attention = 'attention' in layer_name
        is_ffn = 'intermediate' in layer_name or ('output.dense' in layer_name and 'attention' not in layer_name)
        
        if is_attention:
            importance = attn_importance[layer_idx]
        elif is_ffn:
            importance = ffn_importance[layer_idx]
        else:
            importance = 0.5  # default
        
        # Linear mapping: importance -> rank
        rank = int(min_rank + importance * (max_rank - min_rank))
        rank = max(min_rank, min(max_rank, rank))
        rank_dict[layer_name] = rank
    
    return rank_dict


# Design ranks for several compression targets
compression_configs = {
    'informed_light': {'min_rank': 200, 'max_rank': 700},
    'informed_moderate': {'min_rank': 64, 'max_rank': 512},
    'informed_aggressive': {'min_rank': 32, 'max_rank': 256},
}

informed_ranks = {}
for config_name, params in compression_configs.items():
    ranks = design_informed_ranks(
        attn_importance, ffn_importance,
        min_rank=params['min_rank'], max_rank=params['max_rank']
    )
    informed_ranks[config_name] = ranks
    
    # Print summary
    rank_values = list(ranks.values())
    print(f"\n{config_name}: min={min(rank_values)}, max={max(rank_values)}, "
          f"mean={np.mean(rank_values):.0f}, std={np.std(rank_values):.0f}")

## 3. Grand Comparison: Uniform vs Adaptive vs Informed

In [ ]:
@torch.no_grad()
def evaluate_model(model, dataset, data_collator, device, batch_size=64):
    """Full evaluation: macro F1, micro F1, per-emotion F1."""
    loader = DataLoader(dataset, batch_size=batch_size, collate_fn=data_collator, shuffle=False)
    all_preds = []
    all_labels = []
    
    model.eval()
    for batch in tqdm(loader, desc="Evaluating", leave=False):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels']
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.cpu()
        preds = (torch.sigmoid(logits) > 0.5).float()
        
        all_preds.append(preds)
        all_labels.append(labels)
    
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    micro_f1 = f1_score(all_labels, all_preds, average='micro', zero_division=0)
    per_emotion = [f1_score(all_labels[:, i], all_preds[:, i], zero_division=0) 
                   for i in range(all_labels.shape[1])]
    
    return {
        'macro_f1': macro_f1,
        'micro_f1': micro_f1,
        'per_emotion_f1': np.array(per_emotion)
    }

In [ ]:
# Evaluate baseline
print("Evaluating baseline...")
baseline_results = evaluate_model(baseline_model, test_ds, data_collator, device)
baseline_params = count_parameters(baseline_model)
print(f"Baseline: macro_f1={baseline_results['macro_f1']:.4f}, "
      f"micro_f1={baseline_results['micro_f1']:.4f}, params={baseline_params:,}")

In [ ]:
# Define all compression strategies to compare
strategies = []

# Strategy 1: Uniform compression at various ranks
for rank in [512, 384, 256, 128, 64, 32]:
    strategies.append({
        'name': f'uniform_r{rank}',
        'type': 'uniform',
        'rank': rank
    })

# Strategy 2: Adaptive (energy-based) at various thresholds
for threshold in [0.99, 0.95, 0.90, 0.80]:
    strategies.append({
        'name': f'adaptive_e{int(threshold*100)}',
        'type': 'adaptive',
        'threshold': threshold
    })

# Strategy 3: Informed compression
for config_name, ranks in informed_ranks.items():
    strategies.append({
        'name': config_name,
        'type': 'informed',
        'ranks': ranks
    })

print(f"Total strategies to evaluate: {len(strategies)}")
for s in strategies:
    print(f"  - {s['name']} ({s['type']})")

In [ ]:
# Evaluate all strategies
all_results = []

all_layers = get_target_layer_names(baseline_model)

for strategy in tqdm(strategies, desc="Evaluating strategies"):
    print(f"\n--- {strategy['name']} ---")
    
    if strategy['type'] == 'uniform':
        compressed = apply_svd_compression(
            baseline_model, rank=strategy['rank'], layer_names=all_layers
        )
    elif strategy['type'] == 'adaptive':
        energy_info = compute_singular_value_energy(baseline_model)
        adaptive_ranks = compute_adaptive_ranks(energy_info, threshold=strategy['threshold'])
        compressed = apply_svd_compression(
            baseline_model, rank=adaptive_ranks, layer_names=all_layers
        )
    elif strategy['type'] == 'informed':
        compressed = apply_svd_compression(
            baseline_model, rank=strategy['ranks'], layer_names=all_layers
        )
    
    compressed.to(device)
    comp_params = count_parameters(compressed)
    comp_ratio = comp_params / baseline_params
    
    results = evaluate_model(compressed, test_ds, data_collator, device)
    
    result_entry = {
        'strategy': strategy['name'],
        'type': strategy['type'],
        'params': comp_params,
        'compression_ratio': comp_ratio,
        'macro_f1': results['macro_f1'],
        'micro_f1': results['micro_f1'],
        'f1_retention': results['macro_f1'] / baseline_results['macro_f1'],
        'per_emotion_f1': results['per_emotion_f1']
    }
    all_results.append(result_entry)
    
    print(f"  params={comp_params:,} ({comp_ratio:.2%}), "
          f"macro_f1={results['macro_f1']:.4f} ({result_entry['f1_retention']:.2%} retained)")
    
    del compressed
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'per_emotion_f1'} for r in all_results])
results_df = results_df.sort_values('compression_ratio')

print("\nAll Results:")
print(results_df[['strategy', 'type', 'compression_ratio', 'macro_f1', 'f1_retention']].to_string(index=False))

## 4. Pareto Frontier

In [ ]:
def compute_pareto_frontier(df, x_col='compression_ratio', y_col='macro_f1'):
    """Find Pareto-optimal points (lower compression_ratio AND higher F1)."""
    pareto = []
    sorted_df = df.sort_values(x_col)
    best_y = -np.inf
    for _, row in sorted_df.iterrows():
        if row[y_col] > best_y:
            pareto.append(row)
            best_y = row[y_col]
    return pd.DataFrame(pareto)


fig, ax = plt.subplots(figsize=(12, 7))

# Color by strategy type
colors = {'uniform': 'steelblue', 'adaptive': 'orange', 'informed': 'crimson'}
markers = {'uniform': 'o', 'adaptive': 's', 'informed': '*'}
sizes = {'uniform': 80, 'adaptive': 100, 'informed': 200}

for stype in ['uniform', 'adaptive', 'informed']:
    subset = results_df[results_df['type'] == stype]
    ax.scatter(
        subset['compression_ratio'], subset['macro_f1'],
        c=colors[stype], marker=markers[stype], s=sizes[stype],
        label=stype.capitalize(), alpha=0.8, edgecolors='white', linewidth=0.5,
        zorder=3
    )
    # Label points
    for _, row in subset.iterrows():
        ax.annotate(
            row['strategy'].replace('uniform_', '').replace('adaptive_', '').replace('informed_', ''),
            (row['compression_ratio'], row['macro_f1']),
            fontsize=7, ha='center', va='bottom', textcoords='offset points', xytext=(0, 5)
        )

# Pareto frontier
pareto_df = compute_pareto_frontier(results_df)
pareto_sorted = pareto_df.sort_values('compression_ratio')
ax.plot(pareto_sorted['compression_ratio'], pareto_sorted['macro_f1'],
        'k--', alpha=0.4, label='Pareto frontier', zorder=1)

# Baseline reference
ax.axhline(y=baseline_results['macro_f1'], color='green', linestyle=':', alpha=0.5, label='Baseline')

ax.set_xlabel('Parameter Ratio (compressed / original)')
ax.set_ylabel('Macro F1')
ax.set_title('Compression Strategy Comparison — Pareto Frontier')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'pareto_frontier_all_strategies.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Per-Emotion Impact Comparison

In [ ]:
# Compare per-emotion F1 for strategies at similar compression levels
# Pick one uniform, one adaptive, one informed at ~similar compression ratio

def find_closest_strategy(results, strategy_type, target_ratio):
    """Find strategy of given type closest to target compression ratio."""
    subset = [r for r in results if r['type'] == strategy_type]
    if not subset:
        return None
    return min(subset, key=lambda r: abs(r['compression_ratio'] - target_ratio))

# Find comparable strategies (target ~60% compression ratio)
target_ratio = 0.6
comparable = {}
for stype in ['uniform', 'adaptive', 'informed']:
    result = find_closest_strategy(all_results, stype, target_ratio)
    if result:
        comparable[stype] = result
        print(f"{stype}: {result['strategy']} (ratio={result['compression_ratio']:.3f}, F1={result['macro_f1']:.4f})")

if len(comparable) >= 2:
    fig, ax = plt.subplots(figsize=(16, 6))
    
    x = np.arange(len(emotion_names))
    width = 0.25
    
    for i, (stype, result) in enumerate(comparable.items()):
        f1_drop = baseline_results['per_emotion_f1'] - result['per_emotion_f1']
        ax.bar(x + i * width - width, f1_drop, width, 
               label=f"{stype} ({result['strategy']})", color=colors.get(stype, 'gray'), alpha=0.8)
    
    ax.set_xlabel('Emotion')
    ax.set_ylabel('F1 Drop (positive = degradation)')
    ax.set_title('Per-Emotion F1 Degradation by Compression Strategy')
    ax.set_xticks(x)
    ax.set_xticklabels(emotion_names, rotation=45, ha='right', fontsize=8)
    ax.legend()
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'per_emotion_strategy_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()

## 6. Fine-Tuning Recovery Analysis

Can a few epochs of fine-tuning recover performance lost to compression? The recovery rate itself reveals whether emotion representations are **distributed** (recoverable) or **concentrated** (permanently lost).

In [ ]:
def finetune_compressed(compressed_model, train_ds, val_ds, data_collator, 
                        epochs=3, lr=2e-5, batch_size=32, output_dir=None):
    """Fine-tune a compressed model for a few epochs."""
    if output_dir is None:
        output_dir = os.path.join(PROJECT_ROOT, 'results', 'finetuned_compressed_tmp')
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=64,
        learning_rate=lr,
        warmup_ratio=0.1,
        eval_strategy='epoch',
        save_strategy='no',
        logging_steps=100,
        report_to='none',
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=0,
    )
    
    trainer = Trainer(
        model=compressed_model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    
    trainer.train()
    return compressed_model

In [ ]:
# Fine-tune the best informed compression strategy
# Pick the moderate informed strategy for fine-tuning
ft_strategy_name = 'informed_moderate'
ft_ranks = informed_ranks[ft_strategy_name]

print(f"Fine-tuning {ft_strategy_name}...")
print(f"Rank range: {min(ft_ranks.values())} - {max(ft_ranks.values())}")

# Create fresh compressed model
ft_model = apply_svd_compression(
    baseline_model, rank=ft_ranks, layer_names=get_target_layer_names(baseline_model)
)

# Evaluate before fine-tuning
ft_model.to(device)
pre_ft_results = evaluate_model(ft_model, test_ds, data_collator, device)
print(f"\nBefore fine-tuning: macro_f1={pre_ft_results['macro_f1']:.4f}")

In [ ]:
# Fine-tune
ft_model = finetune_compressed(
    ft_model, train_ds, val_ds, data_collator,
    epochs=3, lr=2e-5
)

# Evaluate after fine-tuning
ft_model.to(device)
post_ft_results = evaluate_model(ft_model, test_ds, data_collator, device)
print(f"\nAfter fine-tuning: macro_f1={post_ft_results['macro_f1']:.4f}")
print(f"Recovery: {pre_ft_results['macro_f1']:.4f} -> {post_ft_results['macro_f1']:.4f} "
      f"(baseline={baseline_results['macro_f1']:.4f})")

In [ ]:
# Per-emotion recovery analysis
recovery_df = pd.DataFrame({
    'emotion': emotion_names,
    'baseline_f1': baseline_results['per_emotion_f1'],
    'compressed_f1': pre_ft_results['per_emotion_f1'],
    'finetuned_f1': post_ft_results['per_emotion_f1'],
})

recovery_df['compression_drop'] = recovery_df['baseline_f1'] - recovery_df['compressed_f1']
recovery_df['ft_recovery'] = recovery_df['finetuned_f1'] - recovery_df['compressed_f1']
recovery_df['residual_gap'] = recovery_df['baseline_f1'] - recovery_df['finetuned_f1']
recovery_df['recovery_rate'] = np.where(
    recovery_df['compression_drop'] > 0.01,
    recovery_df['ft_recovery'] / recovery_df['compression_drop'],
    np.nan
)

print("\nPer-Emotion Recovery Analysis:")
print(recovery_df[['emotion', 'baseline_f1', 'compressed_f1', 'finetuned_f1', 
                     'recovery_rate']].to_string(index=False))

In [ ]:
# Visualize recovery
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: bar chart comparing baseline, compressed, fine-tuned
x = np.arange(len(emotion_names))
width = 0.25

axes[0].bar(x - width, recovery_df['baseline_f1'], width, label='Baseline', color='green', alpha=0.7)
axes[0].bar(x, recovery_df['compressed_f1'], width, label='Compressed', color='red', alpha=0.7)
axes[0].bar(x + width, recovery_df['finetuned_f1'], width, label='Fine-tuned', color='blue', alpha=0.7)
axes[0].set_xlabel('Emotion')
axes[0].set_ylabel('F1 Score')
axes[0].set_title('Per-Emotion: Baseline vs Compressed vs Fine-tuned')
axes[0].set_xticks(x)
axes[0].set_xticklabels(emotion_names, rotation=45, ha='right', fontsize=7)
axes[0].legend(fontsize=8)

# Right: recovery rate (emotions sorted by recovery)
valid_recovery = recovery_df.dropna(subset=['recovery_rate']).sort_values('recovery_rate')
colors_recovery = ['red' if r < 0.5 else 'orange' if r < 0.8 else 'green' 
                    for r in valid_recovery['recovery_rate']]
axes[1].barh(range(len(valid_recovery)), valid_recovery['recovery_rate'].clip(-0.5, 1.5),
             color=colors_recovery, alpha=0.8)
axes[1].set_yticks(range(len(valid_recovery)))
axes[1].set_yticklabels(valid_recovery['emotion'], fontsize=8)
axes[1].axvline(x=1.0, color='green', linestyle='--', alpha=0.5, label='Full recovery')
axes[1].axvline(x=0.0, color='red', linestyle='--', alpha=0.5, label='No recovery')
axes[1].set_xlabel('Recovery Rate')
axes[1].set_title('Fine-Tuning Recovery Rate per Emotion')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'finetuning_recovery.png'), dpi=150, bbox_inches='tight')
plt.show()

# Interpretation
low_recovery = valid_recovery[valid_recovery['recovery_rate'] < 0.5]
high_recovery = valid_recovery[valid_recovery['recovery_rate'] > 0.8]
print(f"\nEmotions with LOW recovery (<50%): {list(low_recovery['emotion'])}")
print(f"  -> These have CONCENTRATED representations (compression is destructive)")
print(f"\nEmotions with HIGH recovery (>80%): {list(high_recovery['emotion'])}")
print(f"  -> These have DISTRIBUTED representations (compression is recoverable)")

## 7. Inference Benchmarks

In [ ]:
def benchmark_inference(model, tokenizer_name='bert-base-uncased', 
                        n_warmup=10, n_runs=50, batch_size=1, seq_len=128, device='cpu'):
    """Benchmark inference latency and throughput."""
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    
    # Create dummy input
    dummy_text = "This is a test sentence for benchmarking inference speed." * 5
    inputs = tokenizer(dummy_text, return_tensors='pt', padding='max_length', 
                       truncation=True, max_length=seq_len)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Expand to batch
    if batch_size > 1:
        inputs = {k: v.expand(batch_size, -1) for k, v in inputs.items()}
    
    model.to(device)
    model.eval()
    
    # Warmup
    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(**inputs)
    
    # Benchmark
    if device != 'cpu' and torch.cuda.is_available():
        torch.cuda.synchronize()
    
    latencies = []
    with torch.no_grad():
        for _ in range(n_runs):
            start = time.perf_counter()
            _ = model(**inputs)
            if device != 'cpu' and torch.cuda.is_available():
                torch.cuda.synchronize()
            latencies.append(time.perf_counter() - start)
    
    latencies = np.array(latencies) * 1000  # convert to ms
    throughput = batch_size / (np.mean(latencies) / 1000)  # samples/sec
    
    return {
        'mean_latency_ms': np.mean(latencies),
        'std_latency_ms': np.std(latencies),
        'p50_latency_ms': np.percentile(latencies, 50),
        'p99_latency_ms': np.percentile(latencies, 99),
        'throughput_sps': throughput
    }

In [ ]:
# Benchmark baseline and best informed
bench_device = 'cpu'  # benchmark on CPU for fair comparison

print("Benchmarking on CPU (batch_size=1, seq_len=128)...")

# Baseline
baseline_bench = benchmark_inference(baseline_model, device=bench_device)
print(f"\nBaseline: {baseline_bench['mean_latency_ms']:.1f}ms "
      f"(+/- {baseline_bench['std_latency_ms']:.1f}ms), "
      f"{baseline_bench['throughput_sps']:.1f} samples/s")

# Compressed (informed moderate)
informed_model = apply_svd_compression(
    baseline_model, rank=informed_ranks['informed_moderate'],
    layer_names=get_target_layer_names(baseline_model)
)
informed_bench = benchmark_inference(informed_model, device=bench_device)
print(f"Informed: {informed_bench['mean_latency_ms']:.1f}ms "
      f"(+/- {informed_bench['std_latency_ms']:.1f}ms), "
      f"{informed_bench['throughput_sps']:.1f} samples/s")

speedup = baseline_bench['mean_latency_ms'] / informed_bench['mean_latency_ms']
print(f"\nSpeedup: {speedup:.2f}x")

del informed_model

In [ ]:
# GPU benchmark (if available)
if torch.cuda.is_available():
    print("\nBenchmarking on GPU (batch_size=32, seq_len=128)...")
    
    baseline_gpu = benchmark_inference(baseline_model, device='cuda', batch_size=32)
    print(f"Baseline GPU: {baseline_gpu['mean_latency_ms']:.1f}ms, "
          f"{baseline_gpu['throughput_sps']:.0f} samples/s")
    
    informed_model = apply_svd_compression(
        baseline_model, rank=informed_ranks['informed_moderate'],
        layer_names=get_target_layer_names(baseline_model)
    )
    informed_gpu = benchmark_inference(informed_model, device='cuda', batch_size=32)
    print(f"Informed GPU: {informed_gpu['mean_latency_ms']:.1f}ms, "
          f"{informed_gpu['throughput_sps']:.0f} samples/s")
    
    gpu_speedup = baseline_gpu['mean_latency_ms'] / informed_gpu['mean_latency_ms']
    print(f"GPU Speedup: {gpu_speedup:.2f}x")
    
    del informed_model
    torch.cuda.empty_cache()
else:
    print("GPU not available, skipping GPU benchmarks.")

## 8. Final Summary Figure

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Panel A: Pareto frontier
ax = axes[0, 0]
for stype in ['uniform', 'adaptive', 'informed']:
    subset = results_df[results_df['type'] == stype]
    ax.scatter(1 - subset['compression_ratio'], subset['macro_f1'],
              c=colors[stype], marker=markers[stype], s=sizes[stype],
              label=stype.capitalize(), alpha=0.8, edgecolors='white', linewidth=0.5)
ax.axhline(y=baseline_results['macro_f1'], color='green', linestyle=':', alpha=0.5)
ax.set_xlabel('Compression (1 - param ratio)')
ax.set_ylabel('Macro F1')
ax.set_title('A) Pareto Frontier')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel B: Layer importance
ax = axes[0, 1]
x = np.arange(12)
width = 0.35
ax.bar(x - width/2, attn_importance, width, label='Attention', color='steelblue', alpha=0.8)
ax.bar(x + width/2, ffn_importance, width, label='FFN', color='coral', alpha=0.8)
ax.set_xlabel('Layer')
ax.set_ylabel('Importance')
ax.set_title('B) Mechanistic Layer Importance')
ax.legend(fontsize=8)
ax.set_xticks(x)

# Panel C: Recovery rates
ax = axes[1, 0]
if len(valid_recovery) > 0:
    sorted_rec = valid_recovery.sort_values('recovery_rate', ascending=True)
    colors_rec = ['red' if r < 0.5 else 'orange' if r < 0.8 else 'green' 
                  for r in sorted_rec['recovery_rate']]
    ax.barh(range(len(sorted_rec)), sorted_rec['recovery_rate'].clip(-0.5, 1.5),
            color=colors_rec, alpha=0.8)
    ax.set_yticks(range(len(sorted_rec)))
    ax.set_yticklabels(sorted_rec['emotion'], fontsize=7)
    ax.axvline(x=1.0, color='green', linestyle='--', alpha=0.5)
    ax.axvline(x=0.0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Recovery Rate')
ax.set_title('C) Fine-Tuning Recovery per Emotion')

# Panel D: Rank allocation visualization
ax = axes[1, 1]
mod_ranks = informed_ranks['informed_moderate']
rank_by_layer_attn = []
rank_by_layer_ffn = []
for l in range(12):
    attn_ranks = [r for name, r in mod_ranks.items() if f'layer.{l}.' in name and 'attention' in name]
    ffn_ranks_l = [r for name, r in mod_ranks.items() if f'layer.{l}.' in name and 'attention' not in name]
    rank_by_layer_attn.append(np.mean(attn_ranks) if attn_ranks else 0)
    rank_by_layer_ffn.append(np.mean(ffn_ranks_l) if ffn_ranks_l else 0)

x = np.arange(12)
ax.bar(x - 0.175, rank_by_layer_attn, 0.35, label='Attention', color='steelblue', alpha=0.8)
ax.bar(x + 0.175, rank_by_layer_ffn, 0.35, label='FFN', color='coral', alpha=0.8)
ax.set_xlabel('Layer')
ax.set_ylabel('SVD Rank')
ax.set_title('D) Informed Rank Allocation (Moderate)')
ax.legend(fontsize=8)
ax.set_xticks(x)

plt.suptitle('Informed Compression: Final Summary', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'informed_compression_summary.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save all final results
results_df.to_csv(os.path.join(results_dir, 'compression_comparison.csv'), index=False)
recovery_df.to_csv(os.path.join(results_dir, 'finetuning_recovery.csv'), index=False)

# Save informed ranks
for config_name, ranks in informed_ranks.items():
    rank_df = pd.DataFrame([
        {'layer_name': name, 'rank': rank} 
        for name, rank in ranks.items()
    ])
    rank_df.to_csv(os.path.join(results_dir, f'{config_name}_ranks.csv'), index=False)

print("All results saved!")
print(f"\n{'='*70}")
print("FINAL SUMMARY")
print(f"{'='*70}")
print(f"Baseline: macro_f1={baseline_results['macro_f1']:.4f}, params={baseline_params:,}")
print(f"\nBest strategy per type:")
for stype in ['uniform', 'adaptive', 'informed']:
    subset = results_df[results_df['type'] == stype]
    best = subset.loc[subset['macro_f1'].idxmax()]
    print(f"  {stype}: {best['strategy']} -> F1={best['macro_f1']:.4f} "
          f"({best['f1_retention']:.1%} retained at {best['compression_ratio']:.1%} params)")
print(f"\nFine-tuning recovery: {pre_ft_results['macro_f1']:.4f} -> {post_ft_results['macro_f1']:.4f}")
print(f"CPU speedup: {speedup:.2f}x")